# Colab — ChickenAct + VideoMAE-Small (PBL Monitoring Kesehatan Ayam, Stage 2)

Fine-tuning **VideoMAE-Small** untuk klasifikasi perilaku ayam (Stage 2),
dari klip **ChickenAct** (ChickenVerse). Output notebook ini: checkpoint
model perilaku di Drive, siap dipakai `inference.py` Stage 2.

## Akun — PENTING, baca dulu
- Training ini jalan di **Colab dengan akun `pandumanis72@gmail.com`**
  (BUKAN `pandunugrahaputra@gmail.com` — akun lama sudah hampir penuh,
  jangan dipakai lagi untuk Stage 2).
- Saat cell mount Drive meminta login Google, **pilih akun
  `pandumanis72@gmail.com`**. Checkpoint di-sync ke Drive akun itu
  (`MyDrive/PBL-Ayam/runs/...`), pola 120 detik seperti Stage 1.
- Tidak ada file ID Drive yang di-hardcode di notebook ini — semua path
  berupa nama folder di bawah `MyDrive/PBL-Ayam`.

## Dataset — dua cara (pilih yang sesuai; notebook menangani keduanya)
- **(A) ChickenAct.zip sudah ada di Drive akun baru** (mis. kamu sudah
  upload dari komputer ke `MyDrive/PBL-Ayam/dataset/` atau folder lain):
  notebook menemukannya berdasarkan **nama file** dan menyalinnya ke sesi
  ini. Tidak perlu tahu foldernya di mana.
- **(B) Belum ada di Drive:** notebook mengunduh langsung dari **Zenodo**
  (record 20672799, ~2 GB, ±10 menit) ke sesi Colab ini — kamu tidak
  wajib upload file 2 GB secara manual. Kalau `SAVE_ZIP_TO_DRIVE = True`,
  zip hasil unduhan juga disalin ke Drive akun baru supaya sesi
  berikutnya bisa pakai cara (A) dan tidak unduh ulang.
- Zip diverifikasi (ukuran 1.974.664.090 byte + md5
  `fa5962dfd9cf99df1b08df613891322c`) sebelum dipakai.

## Kelas yang dilatih — baca, ini penting
- ChickenAct berisi **15 kelas asli** (diperiksa langsung dari zip):
  feeding, drinking, litter_pecking, litter_scratching, walking, running,
  wing_flapping, resting, standing, preening, dust_bathing, stretching,
  head_scratching, body_shaking, pooping. Daftar ini **berbeda** dari
  daftar whitepaper (tidak ada sitting / fighting / pecking_other /
  distress di data).
- Pemetaan ke kelas sistem (lihat `data_prep/label_mapping.py`):
  **INGESTIVE** (feeding, drinking, litter_pecking, litter_scratching),
  **ACTIVE** (walking, running, wing_flapping), **RESTING** (resting,
  standing, preening, dust_bathing, stretching, head_scratching,
  body_shaking). **Pooping dikecualikan** (260 klip — peristiwa
  fisiologis sesaat, bukan state perilaku).
- **AGONISTIC tidak dilatih di sini** karena tidak ada satu pun klip
  agresi/distress di ChickenAct. Model ini 3 kelas. Id AGONISTIC
  dicadangkan di taksonomi; kalau nanti ada data agresi, pipeline yang
  sama melatih 4 kelas tanpa perubahan kode. Deteksi agresi di Stage 3
  untuk sementara mengandalkan fitur lain (variansi kecepatan, overlap).

## Alur notebook
1. Cek GPU + mount Drive (akun baru) — 2. Konfigurasi — 3. Dataset
   (cara A / B) — 4. Unzip + cek struktur — 5. Clone repo + install
   dependensi — 6. Data prep (mapping + split train/val/test) —
   7. Ekstrak 16 frame/klip — 8. Training (sync checkpoint ke Drive tiap
   120 detik, bisa resume) — 9. Baca metrik.


In [ ]:
# 0. Cek GPU + mount Drive (login dengan akun pandumanis72@gmail.com)
import torch
print('torch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('PERINGATAN: GPU tidak terdeteksi. Aktifkan Runtime > Change runtime type > T4 GPU, lalu run ulang.')

from google.colab import drive
drive.mount('/content/drive')  # <-- pilih akun pandumanis72@gmail.com
print('Drive termount di /content/drive')


In [ ]:
# 1. Konfigurasi — ubah hanya di sini kalau perlu
EPOCHS = 10
RUN_NAME = 'stage2-videomae'  # folder run di Drive (sudah disiapkan)
HF_CHECKPOINT = 'MCG-NJU/videomae-small-finetuned-kinetics'
# Catatan checkpoint: 'MCG-NJU/videomae-small' (self-supervised saja) TIDAK
# ADA di Hugging Face. Yang tersedia untuk varian Small adalah checkpoint
# di atas (ViT-Small ~22M parameter, sudah fine-tune Kinetics-400).
# Bobot backbone-nya ditransfer; classifier head di-init ulang untuk
# kelas perilaku (train.py).
LR = 5e-5
TRAIN_BATCH = 16
EVAL_BATCH = 32
SEED = 42

# Dataset: dua cara (A dari Drive, B unduh Zenodo). Jangan isi file ID.
ZENODO_URL = 'https://zenodo.org/api/records/20672799/files/ChickenAct.zip/content'
EXPECTED_SIZE = 1974664090
EXPECTED_MD5 = 'fa5962dfd9cf99df1b08df613891322c'
SAVE_ZIP_TO_DRIVE = True  # kalau cara (B), salin juga zip ke Drive (sesi berikut pakai cara A)

DRIVE_ROOT = '/content/drive/MyDrive/PBL-Ayam'
DRIVE_DATASET_DIR = f'{DRIVE_ROOT}/dataset'
DRIVE_RUNS_DIR = f'{DRIVE_ROOT}/runs'
LOCAL_ROOT = '/content/chickenact_work'
LOCAL_ZIP = f'{LOCAL_ROOT}/ChickenAct.zip'
EXTRACT_DIR = f'{LOCAL_ROOT}/ChickenAct'
PREPARED_DIR = f'{LOCAL_ROOT}/prepared'
FRAMES_DIR = f'{LOCAL_ROOT}/frames'
LOCAL_OUTPUT = f'/content/videomae_output/{RUN_NAME}'
DRIVE_RUN_DIR = f'{DRIVE_RUNS_DIR}/{RUN_NAME}'
REPO_DIR = '/content/chicken-anomaly-system'
STAGE2_DIR = f'{REPO_DIR}/backend/stage2_behavior'

import os
for d in (DRIVE_DATASET_DIR, DRIVE_RUNS_DIR, LOCAL_ROOT, DRIVE_RUN_DIR):
    os.makedirs(d, exist_ok=True)
print('DRIVE_ROOT   :', DRIVE_ROOT)
print('LOCAL_ZIP    :', LOCAL_ZIP)
print('DRIVE_RUN_DIR:', DRIVE_RUN_DIR)
print('LOCAL_OUTPUT :', LOCAL_OUTPUT)


In [ ]:
# 2. Dataset — cara (A) dari Drive (cari berdasarkan nama file),
#    kalau tidak ada, cara (B) unduh langsung dari Zenodo.
import os, shutil, subprocess, hashlib

def md5_of_file(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

# --- Cara (A): cari ChickenAct.zip di Drive akun ini (nama file saja) ---
drive_zip = None
candidate = os.path.join(DRIVE_DATASET_DIR, 'ChickenAct.zip')
if os.path.exists(candidate):
    drive_zip = candidate
else:
    print('Tidak ada di', DRIVE_DATASET_DIR, '- mencari di seluruh Drive ...')
    found = subprocess.run(
        ['find', '/content/drive/MyDrive', '-maxdepth', '4',
         '-name', 'ChickenAct.zip'],
        capture_output=True, text=True).stdout.splitlines()
    found = [p for p in found if os.path.exists(p)]
    if found:
        drive_zip = found[0]
        print('Ditemukan di Drive:', drive_zip)

if drive_zip:
    print('Cara (A): pakai zip dari Drive:', drive_zip,
          os.path.getsize(drive_zip), 'byte')
    if not (os.path.exists(LOCAL_ZIP)
            and os.path.getsize(LOCAL_ZIP) == os.path.getsize(drive_zip)):
        print('Menyalin zip dari Drive ke sesi ini ...')
        shutil.copyfile(drive_zip, LOCAL_ZIP)
    else:
        print('Salinan lokal sudah ada.')
else:
    # --- Cara (B): unduh langsung dari Zenodo ke sesi Colab ---
    print('Cara (B): zip belum ada di Drive — unduh langsung dari Zenodo ...')
    print('URL:', ZENODO_URL)
    if os.path.exists(LOCAL_ZIP) and os.path.getsize(LOCAL_ZIP) == EXPECTED_SIZE:
        print('File lokal sudah lengkap, lanjut verifikasi.')
    else:
        r = subprocess.run(['curl', '-L', '-C', '-', '-o', LOCAL_ZIP, ZENODO_URL])
        assert r.returncode == 0, 'Unduh Zenodo gagal.'
    if SAVE_ZIP_TO_DRIVE:
        dst = os.path.join(DRIVE_DATASET_DIR, 'ChickenAct.zip')
        if not (os.path.exists(dst) and os.path.getsize(dst) == EXPECTED_SIZE):
            print('Menyalin zip hasil unduhan ke Drive (untuk sesi berikut) ...')
            shutil.copyfile(LOCAL_ZIP, dst)
            print('Tersimpan di Drive:', dst)

print('Verifikasi zip:', LOCAL_ZIP, os.path.getsize(LOCAL_ZIP), 'byte')
assert os.path.getsize(LOCAL_ZIP) == EXPECTED_SIZE, (
    f"Ukuran zip {os.path.getsize(LOCAL_ZIP)} != {EXPECTED_SIZE}")
print('md5:', md5_of_file(LOCAL_ZIP))
assert md5_of_file(LOCAL_ZIP) == EXPECTED_MD5, 'md5 zip tidak cocok!'
print('Zip OK (ukuran + md5 cocok).')


In [ ]:
# 3. Unzip + cek struktur dataset
import os, zipfile
from collections import Counter

if not (os.path.isdir(EXTRACT_DIR) and os.listdir(EXTRACT_DIR)):
    print('Ekstrak', LOCAL_ZIP, '->', EXTRACT_DIR, '...')
    with zipfile.ZipFile(LOCAL_ZIP) as zf:
        zf.extractall(EXTRACT_DIR)
    print('Ekstrak selesai.')
else:
    print('Folder ekstrak sudah ada:', EXTRACT_DIR)

# Struktur: satu folder per kelas berisi .mp4 (15 kelas, 15.250 klip).
counts = Counter()
for name in sorted(os.listdir(EXTRACT_DIR)):
    p = os.path.join(EXTRACT_DIR, name)
    if os.path.isdir(p):
        n = sum(1 for f in os.listdir(p) if f.lower().endswith('.mp4'))
        counts[name] = n
print('Folder kelas:', len(counts), '| total klip:',
      sum(counts.values()))
for k, v in sorted(counts.items()):
    print(f'  {k}: {v}')
assert len(counts) == 15 and sum(counts.values()) == 15250, (
    'Struktur dataset tidak sesuai harapan (15 kelas / 15.250 klip).')


In [ ]:
# 4. Clone repo (branch stage2) + install dependensi training
import os, subprocess, sys

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('Repo sudah ada, pull terbaru ...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=False)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout',
                    'stage2/behavior-setup'], check=False)
else:
    subprocess.run(['git', 'clone', '--branch', 'stage2/behavior-setup',
                    'https://github.com/Ysoap/chicken-anomaly-system.git',
                    REPO_DIR], check=True)
    print('Repo ter-clone di', REPO_DIR)

print('Install transformers + accelerate ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.46', 'accelerate'], check=True)  # >=4.46: warmup_ratio baru ada sejak 4.46
import transformers
print('transformers:', transformers.__version__)
print('Script Stage 2 ada:', os.path.exists(
    os.path.join(STAGE2_DIR, 'train.py')))


In [ ]:
# 5. Data prep: mapping 15 -> kelas sistem + split train/val/test
import os, subprocess, sys, json

cmd = [sys.executable, os.path.join(STAGE2_DIR, 'data_prep',
                                    'prepare_chickenact.py'),
       '--zip', LOCAL_ZIP, '--extract-dir', EXTRACT_DIR,
       '--out-dir', PREPARED_DIR,
       '--expected-md5', EXPECTED_MD5]
print('Perintah:', ' '.join(cmd))
r = subprocess.run(cmd)
assert r.returncode == 0, 'Data prep gagal.'
with open(os.path.join(PREPARED_DIR, 'class_distribution.json')) as f:
    dist = json.load(f)
print()
print('Distribusi kelas sistem (train/val/test):')
for k, v in dist['per_label4'].items():
    print(f"  {k}: {v} klip "
          f"(train {dist['per_label4_train'][k]} / "
          f"val {dist['per_label4_val'][k]} / "
          f"test {dist['per_label4_test'][k]})")
print('Dikecualikan:', dist.get('excluded'))
print('Manifest:', os.path.join(PREPARED_DIR, 'manifest.json'))


In [ ]:
# 6. Ekstrak 16 frame per klip (sekali saja; training baca JPEG, cepat)
#    Estimasi: 15.250 klip, beberapa menit sampai ~30 menit (4 worker).
#    Hasil: FRAMES_DIR/frames_manifest.json — dipakai train.py.
import os, subprocess, sys

cmd = [sys.executable, os.path.join(STAGE2_DIR, 'data_prep',
                                    'extract_frames.py'),
       '--manifest', os.path.join(PREPARED_DIR, 'manifest.json'),
       '--data-root', EXTRACT_DIR,
       '--out-dir', FRAMES_DIR, '--workers', '4']
print('Perintah:', ' '.join(cmd))
r = subprocess.run(cmd)
assert r.returncode == 0, 'Ekstrak frame gagal (lihat log di atas).'
print('Manifest frame:', os.path.join(FRAMES_DIR, 'frames_manifest.json'))


## 7. Training — fine-tuning VideoMAE-Small

Menjalankan `train.py` (HF Trainer) sebagai subprocess, dengan **sync
checkpoint ke Drive akun baru tiap 120 detik** (pola yang sama seperti
Stage 1). Yang di-sync: folder checkpoint terbaru (file model + state,
TANPA `optimizer.pt`/`scheduler.pt`/`rng_state.pth` supaya cepat —
konsekuensinya resume akan me-restart optimizer, dicatat di sini) dan
di akhir: folder `best/` + `train_metrics.json`.

**Resume:** kalau sesi Colab putus / limit habis, jalankan lagi notebook
ini dari awal (Run All) — cell training akan menemukan checkpoint di
Drive dan melanjutkan dari situ (bukan dari nol). Jangan hapus folder
run di Drive sebelum training selesai.


In [ ]:
# 7. Training VideoMAE-Small (tuning dari checkpoint Kinetics) + sync Drive
import os, subprocess, threading, time, shutil, sys, glob

os.makedirs(LOCAL_OUTPUT, exist_ok=True)
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)

# --- Resume: cari checkpoint terbaru di Drive (akun baru) ---
resume_arg = None
drive_ckpts = sorted(glob.glob(os.path.join(DRIVE_RUN_DIR, 'checkpoint-*')))
if drive_ckpts:
    latest_drive = drive_ckpts[-1]
    print('Ditemukan checkpoint di Drive, resume dari situ:', latest_drive)
    local_ckpt = os.path.join(LOCAL_OUTPUT, os.path.basename(latest_drive))
    if os.path.isdir(local_ckpt):
        shutil.rmtree(local_ckpt)
    shutil.copytree(latest_drive, local_ckpt)
    resume_arg = 'auto'
else:
    print('Tidak ada checkpoint di Drive, training dari awal '
          '(tuning dari checkpoint Kinetics).')

# --- Sync ke Drive selama training (cek tiap 120 detik) ---
_sync_stop = threading.Event()
_sync_seen = {}

def _copy_file_if_changed(src, dst):
    try:
        if not os.path.exists(src):
            return False
        key = (os.path.getsize(src), os.path.getmtime(src))
        if _sync_seen.get(src) == key:
            return False
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        tmp = dst + '.tmp'
        shutil.copyfile(src, tmp)
        os.replace(tmp, dst)
        _sync_seen[src] = key
        print(f'[sync Drive] {os.path.basename(src)} -> {dst} '
              f'({key[0]} byte)')
        return True
    except Exception as e:
        print(f'[sync Drive] gagal salin {src}: {e}')
        return False

# File model/state yang di-sync dari checkpoint (optimizer dkk dilewati).
_SYNC_FILES = ('config.json', 'model.safetensors', 'pytorch_model.bin',
               'trainer_state.json', 'training_args.bin',
               'generation_config.json', 'preprocessor_config.json')

def _sync_latest_checkpoint():
    ckpts = sorted(glob.glob(os.path.join(LOCAL_OUTPUT, 'checkpoint-*')))
    if not ckpts:
        return
    latest = ckpts[-1]
    dst_dir = os.path.join(DRIVE_RUN_DIR, os.path.basename(latest))
    os.makedirs(dst_dir, exist_ok=True)
    for fn in _SYNC_FILES:
        _copy_file_if_changed(os.path.join(latest, fn),
                              os.path.join(dst_dir, fn))

def _sync_loop():
    while not _sync_stop.is_set():
        _sync_latest_checkpoint()
        _sync_stop.wait(120)

_sync_thread = threading.Thread(target=_sync_loop, daemon=True)
_sync_thread.start()
print('Sync checkpoint ke Drive aktif (cek tiap 120 detik).')

cmd = [sys.executable, os.path.join(STAGE2_DIR, 'train.py'),
       '--frames-manifest', os.path.join(FRAMES_DIR, 'frames_manifest.json'),
       '--data-root', FRAMES_DIR,
       '--checkpoint', HF_CHECKPOINT,
       '--output-dir', LOCAL_OUTPUT,
       '--epochs', str(EPOCHS), '--lr', str(LR),
       '--train-batch', str(TRAIN_BATCH), '--eval-batch', str(EVAL_BATCH),
       '--seed', str(SEED)]
if resume_arg:
    cmd += ['--resume', resume_arg]
    print('Mode: RESUME dari checkpoint Drive')
else:
    print('Mode: TRAINING BARU dari checkpoint Kinetics')
print('Perintah:', ' '.join(cmd))

try:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    print('Training process exit code:', proc.returncode)
finally:
    _sync_stop.set()
    _sync_thread.join(timeout=30)
    _sync_latest_checkpoint()
    # Sync hasil akhir: model terbaik + metrik.
    best_local = os.path.join(LOCAL_OUTPUT, 'best')
    if os.path.isdir(best_local):
        best_drive = os.path.join(DRIVE_RUN_DIR, 'best')
        shutil.copytree(best_local, best_drive, dirs_exist_ok=True)
        print('Model terbaik tersalin ke Drive:', best_drive)
    _copy_file_if_stable = _copy_file_if_changed
    _copy_file_if_stable(os.path.join(LOCAL_OUTPUT, 'train_metrics.json'),
                         os.path.join(DRIVE_RUN_DIR, 'train_metrics.json'))
    print('Sync checkpoint terakhir ke Drive selesai.')

if proc.returncode != 0:
    raise RuntimeError(
        f"Training GAGAL (exit code {proc.returncode}). "
        "Lihat error di output cell ini di atas — jangan lanjut, "
        "perbaiki dulu baru Run All lagi.")

print('Training selesai / sesi ini berakhir.')
print('Model terbaik lokal:', os.path.join(LOCAL_OUTPUT, 'best'))
print('Model terbaik Drive:', os.path.join(DRIVE_RUN_DIR, 'best'))


In [ ]:
# 8. Baca metrik training (akurasi val + per kelas)
import os, json

metrics_path = os.path.join(LOCAL_OUTPUT, 'train_metrics.json')
drive_metrics = os.path.join(DRIVE_RUN_DIR, 'train_metrics.json')
path = metrics_path if os.path.exists(metrics_path) else drive_metrics
print('Metrik:', path)
if os.path.exists(path):
    with open(path) as f:
        m = json.load(f)
    print('Checkpoint awal:', m.get('checkpoint'))
    print('Kelas dilatih:', m.get('classes'), '| num_labels:', m.get('num_labels'))
    print('Epoch:', m.get('epochs'), '| LR:', m.get('lr'))
    print('train_loss:', m.get('train_loss'))
    print('Metrik val (model terbaik):')
    for k, v in m.get('val_metrics_best', {}).items():
        print(f'  {k}: {v}')
else:
    print('train_metrics.json belum ada — training belum selesai / gagal.')
print()
print('Model terbaik lokal ada:', os.path.isdir(os.path.join(LOCAL_OUTPUT, 'best')))
print('Model terbaik Drive ada:', os.path.isdir(os.path.join(DRIVE_RUN_DIR, 'best')))


## Selesai untuk notebook ini — langkah berikutnya (bukan di notebook ini)

1. **Cek metrik di atas.** Kalau akurasi val masuk akal (jauh di atas
   tebakan acak 33% untuk 3 kelas) dan akurasi per kelas tidak ada yang
   0, model perilaku sudah bisa dipakai. Kalau ada kelas yang 0, periksa
   distribusi / sampling frame-nya dulu sebelum lanjut.
2. **Evaluasi test set** (held-out, 14%): jalankan `train.py` dalam mode
   eval atau script evaluasi di `backend/stage2_behavior/` — metrik test
   yang jujur, seperti AP test Stage 1.
3. **Simpan model ke repo** (Git LFS, seperti `best.pth` Stage 1):
   `backend/stage2_behavior/models/` — catat di README Stage 2.
4. Setelah itu baru: uji `inference.py` Stage 2 di video sample
   (`tracking_tests/` Stage 1) — klip per track -> label perilaku;
   lalu Stage 3 (aturan durasi -> aman/dicurigai/sakit + ambang).

Kredit dataset di laporan: ChickenVerse, Amirivojdan dkk., Univ.
Tennessee, lisensi CC BY-NC-SA 4.0, Zenodo DOI 10.5281/zenodo.20672799.
Model: VideoMAE-Small (`MCG-NJU/videomae-small-finetuned-kinetics`),
input 16 frame @ 224x224.
